---
# Scipy
---

## Outils de calculs scientifiques pour Python

Collection d'algorithmes mathématiques et autres fonctions utiles basés sur Numpy.<br/>
Organisé en sous-catégories couvrant différents dommaines scientifiques (signal, stats, cluster, ...)

doc : http://www.scipy.org/

| Subpackage | Description   |
|------|------|
|cluster|Clustering algorithms|
|constants|Physical and mathematical constants|
|<b>fftpack</b>|Fast Fourier Transform routines|
|integrate|Integration and ordinary differential equation solvers|
|<b>interpolate</b>|Interpolation and smoothing splines|
|io|Input and Output|
|linalg|Linear algebra|
|ndimage|N-dimensional image processing|
|odr|Orthogonal distance regression|
|optimize|Optimization and root-finding routines|
|<b>signal</b>|Signal processing|
|sparse|Sparse matrices and associated routines|
|spatial|Spatial data structures and algorithms|
|special|Special functions|
|stats|Statistical distributions and functions|
|weave|C/C++ integration|

In [ ]:
import numpy as np
import scipy
from scipy import interpolate
from scipy import fftpack
from scipy import signal

import matplotlib.pyplot as plt
%matplotlib inline

from scipy.io.wavfile import read

# Exemple 1 : Interpolations

- Voici des données à interpoler.
Nous allons les interpoler de manière linéaire et cubique et tracer le résultat sur une figure.

on va utiliser [scipy.interpolate.interp1d](http://docs.scipy.org/doc/scipy-0.14.0/reference/generated/scipy.interpolate.interp1d.html)

rappel sur les interpolations [ici](https://fr.wikipedia.org/wiki/Interpolation_num%C3%A9rique) 

In [ ]:
#Données :
x = np.linspace(0, 10, num=10)
y = np.cos(-x**2/9.0)

fig, ax = plt.subplots()
ax.plot(x,y, 'o')

In [ ]:
# Interpolations :
f1 = scipy.interpolate.interp1d(x, y)
f2 = scipy.interpolate.interp1d(x, y, 'cubic')

# Tracés :
xnew = np.linspace(0, 10,num=1000)
f1_inter = f1(xnew)
f2_inter = f2(xnew)

fig, ax = plt.subplots()
ax.plot(x, y, 'o', xnew, f1_inter, '-', xnew, f2_inter, '--')
ax.legend(['data', 'linear', 'cubic'], loc='best')

## Exemple 2 : FFT + Filtrage

### Etape 1 : Ouvrir et afficher un son

- On souhaite ouvrir un fichier son ("speech.wav" dans Data), et afficher ce son + le lire.
- Fréquence d'échantillonage du son : 11025 Hz

In [ ]:
# read audio samples
fs, audio = read("./data/speech.wav")
print(audio)
print(fs)
print(audio.shape)
print("duree du son : "+ str(audio.size/fs) + "secondes")

# Création du signal
times = np.arange(0,audio.size/fs , 1./fs, dtype = 'float64')  # Vecteur temps
l = times.size
print(l)

# Figure
fig, ax = plt.subplots(figsize=(20,5)) 
ax.plot(times, audio)
ax.set_xlabel('Time [sec]')
ax.set_ylabel('Amplitude')

In [ ]:
import IPython.display
from ipywidgets import interact, interactive, fixed

IPython.display.Audio(data=audio, rate=fs/1, autoplay=False)
# IPython.display.Audio(data=audio, rate=fs/2)
# IPython.display.Audio(data=audio, rate=fs/0.5)


### Etape 2 : Calcul et tracé du spectre

- Pour voir le contenu fréquentiel de notre signal, nous allons calculer le "spectre" du signal et le tracer. <br/> 
La FFT (Fast Fourier Transform) est une transformation mathématique qui donne une vision de ce spectre de manière "assez simple"

In [ ]:
# Calcul du spectre
f, Pxx_den = signal.welch(audio, fs, nperseg=1024)

# Tracé
fig, ax = plt.subplots(figsize=(10,5)) 
ax.semilogy(f, Pxx_den)   #  plot with log scaling on the y axis
ax.set_xlabel('frequency [Hz]')
ax.set_ylabel('PSD [V**2/Hz]')


### Etape 3 : Filtrage du signal

- Notre signal est bruité.. On va nettoyer le signal à l'aide d'un filtre qui enlève toutes les fréquences > 4000 Hz et < 100 Hz (passe-bande)

1. Création du filtre : 

In [ ]:
# Paramètres de notre filtre :
f_lowcut = 100.
f_hicut = 4000.

print(fs)
nyq = 0.5 * fs
N = 6                 # Ordre du filtre
Wn = [f_lowcut/nyq,f_hicut/nyq]  # Nyquist frequency fraction
print(Wn)

# Création du filtre :
b, a = scipy.signal.butter(N, Wn, 'band')
#b, a = scipy.signal.butter(N, f_hicut/nyq, 'lowpass')
print(a)
print(b)

# Calcul de la reponse en fréquence du filtre
w, h = signal.freqz(b, a)

# Tracé de la réponse en fréquence du filtre
fig, ax = plt.subplots(figsize=(8,5)) 

ax.plot(0.5*fs*w/np.pi, np.abs(h), 'b')

ax.set_xlabel('frequency [Hz]')
ax.set_ylabel('Amplitude [dB]')
ax.grid(which='both', axis='both')


2 : On applique ce filtre à notre signal

In [ ]:
# Applique le filtre au signal :
filtered_sig = scipy.signal.filtfilt(b, a, audio)

# Tracé du signal filtré
fig, ax = plt.subplots(figsize=(15,5)) 
ax.plot(times, filtered_sig, 'r')
ax.set_xlabel('Temps [sec]')
ax.set_ylabel('Amplitude')

In [ ]:
#jouons ce son filtré : 
IPython.display.Audio(data=filtered_sig, rate=fs/1)

In [ ]:
#pour comparaison, signal original
IPython.display.Audio(data=audio, rate=fs/1)

### Etape 4 : Tracé du spectre du signal filtré

In [ ]:
# Calcul du spectre
f, Pxx_den_filered = signal.welch(filtered_sig, fs, nperseg=1024)

# Tracé
fig, ax = plt.subplots(figsize=(10,5)) 
ax.semilogy(f, Pxx_den, color='g')
ax.semilogy(f, Pxx_den_filered, color='r')
ax.set_xlabel('frequency [Hz]')
ax.set_ylabel('PSD [V**2/Hz]')

# Bonus : Différence entre `lfilter` et `filtfilt`

- `lfilter` applique le filtre **une seule fois**, ce qui peut introduire un **décalage (retard de phase)** dans le signal.
- `filtfilt` applique le filtre **dans les deux sens (aller-retour)**, ce qui **annule le décalage** et donne un résultat plus propre.
- `filtfilt` est donc souvent préférable pour un **traitement de signal "sans distorsion de phase"**.


In [ ]:
# Applique le filtre au signal :
fs = 1000.
times = np.arange(1000)/fs
sig = np.sin(2*np.pi*50.*times) + np.sin(2*np.pi*10.*times)

# creation du filtre
f1, f2 = 40., 100.  # Hz
Wn = [f1/fs*2.,f2/fs*2.]  # Nyquist frequency fraction
b, a = scipy.signal.butter(8, Wn, 'bandstop')


sig_lfilter = scipy.signal.lfilter(b, a, sig, axis=0)
sig_filtfilt = scipy.signal.filtfilt(b, a, sig)

# Tracé du signal filtré
fig, ax = plt.subplots(figsize=(15,5))
ax.plot(times, sig_lfilter, 'r',label='sig_lfilter')
ax.plot(times, sig_filtfilt, 'g', label='sig_filtfilt')
ax.plot(times, sig, 'k', ls='--', lw=2, label='sig')

ax.set_xlabel('Temps [sec]')
ax.set_ylabel('Amplitude')
ax.legend()
#ax.set_xlim(0.2, 0.5)